In [ ]:
import csv
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException
import pandas as pd

### creating the search link

In [ ]:
import csv
import urllib.parse

def create_google_maps_search_link(row, search_columns):
    """
    Create a Google Maps search link from specified columns in the row.
    
    Args:
        row: Dictionary containing row data
        search_columns: List of column names to include in the search
    
    Returns:
        Google Maps search URL
    """
    # Extract values from specified columns and join them
    search_terms = []
    for col in search_columns:
        if col in row and row[col]:
            search_terms.append(str(row[col]).strip())
    
    if not search_terms:
        return ""
    
    # Join search terms with spaces and URL encode
    search_query = " ".join(search_terms)
    encoded_query = urllib.parse.quote(search_query)
    
    # Create Google Maps search URL
    search_url = f"https://www.google.com/maps/search/{encoded_query}"
    
    return search_url

def process_csv(input_file, output_file, search_columns=None):
    """
    Process CSV file and add Google Maps search links.
    
    Args:
        input_file: Path to input CSV file
        output_file: Path to output CSV file
        search_columns: List of column names to use for search link creation.
                       If None, all columns will be used.
    """
    try:
        # Read input CSV
        with open(input_file, 'r', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            original_fieldnames = reader.fieldnames
            
            if original_fieldnames is None:
                print("Error: Input file has no headers")
                return
            
            # Use all columns if search_columns not specified
            if search_columns is None:
                search_columns = list(original_fieldnames)
            
            # Verify all search columns exist in input
            missing_columns = [col for col in search_columns if col not in original_fieldnames]
            if missing_columns:
                print(f"Warning: These columns not found in input: {missing_columns}")
                # Only use columns that exist
                search_columns = [col for col in search_columns if col in original_fieldnames]
            
            # Prepare output fieldnames
            fieldnames = list(original_fieldnames) + ['google_maps_search_link']
            
            # Write output CSV
            with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
                writer = csv.DictWriter(outfile, fieldnames=fieldnames)
                writer.writeheader()
                
                for i, row in enumerate(reader):
                    # Create search link
                    search_link = create_google_maps_search_link(row, search_columns)
                    
                    # Add new column to row
                    row['google_maps_search_link'] = search_link
                    
                    # Write row
                    writer.writerow(row)
                    
                    # Optional: Print progress
                    if (i + 1) % 100 == 0:
                        print(f"Processed {i + 1} rows...")
                
                print(f"Processed {i + 1} total rows")
                
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found")
    except Exception as e:
        print(f"Error processing CSV: {str(e)}")

def process_csv_with_column_detection(input_file, output_file):
    """
    Process CSV file with automatic column detection for search.
    Prioritizes columns with names like 'name', 'address', etc.
    """
    try:
        # Read input CSV to detect columns
        with open(input_file, 'r', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            original_fieldnames = reader.fieldnames
            
            if original_fieldnames is None:
                print("Error: Input file has no headers")
                return
            
            # Prioritize certain column names for search
            priority_patterns = ['name', 'address', 'city', 'place', 'location', 'ort', 'adresse']
            
            # Find columns matching priority patterns
            priority_columns = []
            other_columns = []
            
            for col in original_fieldnames:
                col_lower = col.lower()
                if any(pattern in col_lower for pattern in priority_patterns):
                    priority_columns.append(col)
                else:
                    other_columns.append(col)
            
            # Use priority columns first, fall back to all columns if no matches
            if priority_columns:
                search_columns = priority_columns
                print(f"Using priority columns for search: {search_columns}")
            else:
                search_columns = list(original_fieldnames)
                print(f"No priority columns found. Using all columns: {search_columns}")
            
        # Process with detected columns
        process_csv(input_file, output_file, search_columns)
        
    except Exception as e:
        print(f"Error in column detection: {str(e)}")

# Example usage
if __name__ == "__main__":
    input_csv = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/843_pools.csv"
    output_csv = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/843_pools_with_search_links.csv"
    
    # Option 1: Process with all columns
    process_csv(input_csv, output_csv)
    
    # Option 2: Process with automatic column detection
    # process_csv_with_column_detection(input_csv, output_csv)
    
    # Option 3: Process with specific columns (example)
    # process_csv(input_csv, output_csv, search_columns=['Name', 'Address', 'City'])
    
    print(f"Processing complete! Output saved to {output_csv}")

In [ ]:
def setup_driver():
    """Setup Chrome driver with appropriate options"""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def extract_place_links_from_page(source):
    """Extract all Google Maps place links from page source"""
    # Pattern to match Google Maps place URLs with coordinates
    place_pattern = r'https://www\.google\.com/maps/place/[^"\']+?@(-?\d+\.\d+),(-?\d+\.\d+)'
    
    matches = re.findall(place_pattern, source)
    
    place_links = []
    for match in matches:
        # Reconstruct the full URL (we have the important part with coordinates)
        lat, lng = match
        # Find the full URL in the source
        full_pattern = rf'(https://www\.google\.com/maps/place/[^"\']+?@{lat},{lng}[^"\']*)'
        full_matches = re.findall(full_pattern, source)
        if full_matches:
            place_links.append(full_matches[0])
    
    return place_links

def extract_coordinates_from_place_link(place_link):
    """Extract coordinates from Google Maps place link"""
    if not place_link:
        return None, None
        
    pattern = r'@([-+]?\d+\.\d+),([-+]?\d+\.\d+)'
    match = re.search(pattern, place_link)
    
    if match:
        lat = float(match.group(1))
        lng = float(match.group(2))
        return lat, lng
    return None, None

def search_to_place_link_selenium(search_url, driver):
    """Convert Google search link to place link by scanning entire page source"""
    try:
        print(f"  Navigating to: {search_url}")
        driver.get(search_url)
        
        # Wait for page to load completely
        time.sleep(4)
        
        # Get the entire page source
        page_source = driver.page_source
        current_url = driver.current_url
        
        print(f"  Current URL: {current_url}")
        print(f"  Page source length: {len(page_source)} characters")
        
        # Check if we were redirected directly to a place page
        if '/maps/place/' in current_url and '@' in current_url:
            print("  Direct redirect to place page")
            lat, lng = extract_coordinates_from_place_link(current_url)
            return current_url, lat, lng, 'success_direct_redirect'
        
        # Search entire page source for place links
        place_links = extract_place_links_from_page(page_source)
        
        print(f"  Found {len(place_links)} place links in page source")
        
        if place_links:
            # Use the first place link found (usually the most relevant)
            first_place_link = place_links[0]
            lat, lng = extract_coordinates_from_place_link(first_place_link)
            
            print(f"  Using place link: {first_place_link[:100]}...")
            print(f"  Coordinates: {lat}, {lng}")
            
            return first_place_link, lat, lng, 'success_page_scan'
        
        # Alternative: Look for coordinates in the page even without full place link
        coord_pattern = r'@(-?\d+\.\d+),(-?\d+\.\d+)'
        coord_matches = re.findall(coord_pattern, page_source)
        
        if coord_matches:
            # Take the first coordinate pair
            lat, lng = float(coord_matches[0][0]), float(coord_matches[0][1])
            print(f"  Found coordinates in page: {lat}, {lng}")
            
            # Construct approximate place URL
            search_query = search_url.split('/')[-1].replace('%20', '+')
            approx_place_url = f"https://www.google.com/maps/place/{search_query}/@{lat},{lng},17z/"
            
            return approx_place_url, lat, lng, 'success_coords_only'
        
        return None, None, None, 'no_place_links_found'
            
    except WebDriverException as e:
        return None, None, None, f'driver_error: {str(e)}'
    except Exception as e:
        return None, None, None, f'error: {str(e)}'

def process_csv(input_file, output_file):
    """Process CSV file and add Google Maps URL, latitude, and longitude columns"""
    
    driver = setup_driver()
    
    try:
        # Read input CSV
        with open(input_file, 'r', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            original_fieldnames = reader.fieldnames
            
            # Find the URL column
            url_column = None
            for field in original_fieldnames:
                if 'url' in field.lower() or 'search' in field.lower():
                    url_column = field
                    break
            
            if not url_column:
                url_column = original_fieldnames[0]  # Use first column as fallback
            
            fieldnames = original_fieldnames + ['google_maps_url', 'latitude', 'longitude', 'status']
            
            # Write output CSV
            with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
                writer = csv.DictWriter(outfile, fieldnames=fieldnames)
                writer.writeheader()
                
                for i, row in enumerate(reader):
                    search_url = row.get(url_column)
                    
                    if not search_url:
                        print(f"Row {i+1}: No search URL found in column '{url_column}'")
                        row['google_maps_url'] = ''
                        row['latitude'] = ''
                        row['longitude'] = ''
                        row['status'] = 'no_url'
                        writer.writerow(row)
                        continue
                    
                    print(f"\nProcessing {i+1}: {search_url}")
                    
                    # Get place link and coordinates
                    place_url, lat, lng, status = search_to_place_link_selenium(search_url, driver)
                    
                    # Add new columns to row
                    row['google_maps_url'] = place_url if place_url else ''
                    row['latitude'] = lat if lat is not None else ''
                    row['longitude'] = lng if lng is not None else ''
                    row['status'] = status
                    
                    writer.writerow(row)
                    
                    print(f"  Final Status: {status}")
                    
                    # Be respectful with timing between requests
                    time.sleep(3)
                    
    except Exception as e:
        print(f"Fatal error: {str(e)}")
    finally:
        driver.quit()
        print("Browser closed")

# Enhanced version with multiple fallback patterns
def extract_place_links_enhanced(source):
    """Enhanced extraction with multiple patterns"""
    patterns = [
        # Primary pattern: full Google Maps place URLs
        r'https://www\.google\.com/maps/place/[^"\']+?@(-?\d+\.\d+),(-?\d+\.\d+)[^"\']*',
        # Fallback: any URL containing /maps/place/ and coordinates
        r'/(maps/place/[^"\']+?@(-?\d+\.\d+),(-?\d+\.\d+)[^"\']*)',
        # Coordinates only (we'll construct URL)
        r'@(-?\d+\.\d+),(-?\d+\.\d+)',
    ]
    
    all_matches = []
    
    for pattern in patterns:
        matches = re.findall(pattern, source)
        if matches:
            if pattern == patterns[0]:  # Full URLs
                all_matches.extend(matches)
            elif pattern == patterns[1]:  # Relative URLs
                for match in matches:
                    full_url = f"https://www.google.com{match[0]}"
                    all_matches.append(full_url)
            elif pattern == patterns[2]:  # Just coordinates
                for match in matches:
                    lat, lng = match
                    # Construct approximate URL
                    approx_url = f"https://www.google.com/maps/place/@{lat},{lng},17z/"
                    all_matches.append(approx_url)
    
    return all_matches

# Alternative function using enhanced extraction
def search_to_place_link_enhanced(search_url, driver):
    """Enhanced version with multiple extraction patterns"""
    try:
        print(f"  Navigating to: {search_url}")
        driver.get(search_url)
        time.sleep(4)
        
        page_source = driver.page_source
        current_url = driver.current_url
        
        print(f"  Page source length: {len(page_source)} characters")
        
        # Check direct redirect first
        if '/maps/place/' in current_url and '@' in current_url:
            lat, lng = extract_coordinates_from_place_link(current_url)
            return current_url, lat, lng, 'success_direct_redirect'
        
        # Use enhanced extraction
        place_links = extract_place_links_enhanced(page_source)
        
        print(f"  Found {len(place_links)} place links/coordinates")
        
        if place_links:
            first_link = place_links[0]
            lat, lng = extract_coordinates_from_place_link(first_link)
            
            if lat and lng:
                print(f"  Coordinates: {lat}, {lng}")
                return first_link, lat, lng, 'success_enhanced'
        
        return None, None, None, 'no_place_links_found'
        
    except Exception as e:
        return None, None, None, f'error: {str(e)}'

if __name__ == "__main__":
    input_csv = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/search_url_latlong.csv"  # Replace with your input file name
    output_csv = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/output_search_url_latlong.csv"  # Replace with your desired output file name
    
    process_csv(input_csv, output_csv)
    print(f"Processing complete! Output saved to {output_csv}")

In [ ]:

import csv
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def debug_page_source(url):
    """Simple function to debug what's in the page source"""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(options=chrome_options)
    
    try:
        print(f"Loading: {url}")
        driver.get(url)
        
        # Wait a bit
        import time
        time.sleep(5)
        
        # Get current URL and page source
        current_url = driver.current_url
        page_source = driver.page_source
        
        print(f"Current URL: {current_url}")
        print(f"Page source length: {len(page_source)}")
        
        # Search for place links
        place_links = re.findall(r'https://www\.google\.com/maps/place/[^"\']+', page_source)
        print(f"Found {len(place_links)} place links:")
        
        for i, link in enumerate(place_links[:5]):  # Show first 5
            print(f"  {i+1}: {link}")
            
        # Search for coordinates pattern
        coords = re.findall(r'@(-?\d+\.\d+),(-?\d+\.\d+)', page_source)
        print(f"Found {len(coords)} coordinate patterns:")
        
        for i, coord in enumerate(coords[:5]):
            print(f"  {i+1}: {coord}")
            
        return page_source
        
    finally:
        driver.quit()

# Test with one URL
test_url = "https://www.google.com/maps/search/Freibad%20Hangeweiher%20Am%20Hangeweiher%2032%2C%2052074%20Aachen%2C%20Germany"
debug_page_source(test_url)

### manual retrival of some urls

In [ ]:
import re
import pandas as pd

def extract_lat_lng(url):
    """
    Extracts latitude and longitude from Google Maps URL.
    It looks for the pattern '@lat,lon,'.
    """
    match = re.search(r'@([-0-9.]+),([-0-9.]+)', url)
    if match:
        return match.group(1), match.group(2)
    return None, None


# --- MAIN SCRIPT ---
input_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/gmaps_urls_manual_retrieval.csv"       # your input CSV
output_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/gmaps_urls_manual_retrieval_with_coords.csv"  # output CSV

# Read CSV (expects a column named 'url')
df = pd.read_csv(input_file)

# Extract lat/lng for each row
df['latitude'], df['longitude'] = zip(*df['google_maps_url'].apply(extract_lat_lng))

# Save to CSV
df.to_csv(output_file, index=False)

print(f"Done! Saved extracted coordinates to: {output_file}")


## finding the  place_ids

In [ ]:
import csv
import re
import time
import os
import hashlib
from datetime import datetime
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException

def setup_driver():
    """Setup Chrome driver with appropriate options"""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def is_place_page_url(url):
    """Check if URL is a Google Maps place page"""
    return '/maps/place/' in url and '@' in url

def generate_search_url(pool_name, address):
    """
    Generate Google Maps search URL from pool name and address
    """
    search_query = f"{pool_name} {address}".strip()
    encoded_query = quote(search_query)
    search_url = f"https://www.google.com/maps/search/{encoded_query}"
    return search_url

def save_place_page_html(source, url, index, output_dir="saved_place_pages"):
    """
    Save HTML of Google Maps place page to a file
    """
    try:
        # Create absolute path for output directory
        abs_output_dir = os.path.abspath(output_dir)
        os.makedirs(abs_output_dir, exist_ok=True)
        
        # Create filename from place name and timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Extract place name from URL for better filename
        place_name = "place"
        if '/maps/place/' in url:
            # Extract place name from URL
            match = re.search(r'/place/([^/@]+)', url)
            if match:
                place_name = match.group(1).replace('+', '_').replace('/', '_')[:50]
        
        filename = f"row_{index}_{place_name}_{timestamp}.html"
        filepath = os.path.join(abs_output_dir, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"<!-- URL: {url} -->\n")
            f.write(f"<!-- Saved at: {datetime.now().isoformat()} -->\n")
            f.write(f"<!-- Row index: {index} -->\n")
            f.write(f"<!-- File: {filename} -->\n")
            f.write("\n" + source)
        
        print(f"  ✓ Saved place page HTML to: {filepath}")
        return filepath
    except Exception as e:
        print(f"  ✗ Failed to save HTML: {str(e)}")
        return None

def extract_coordinates_from_url(url):
    """
    Extract coordinates from Google Maps place URL
    """
    if not url:
        return None, None
    
    pattern = r'@([-+]?\d+\.\d+),([-+]?\d+\.\d+)'
    match = re.search(pattern, url)
    
    if match:
        try:
            lat = float(match.group(1))
            lng = float(match.group(2))
            return lat, lng
        except ValueError:
            return None, None
    
    return None, None

def extract_place_id_from_html(html_source):
    """
    Extract Google Maps place ID from HTML source
    Look for pattern: placeid\\u003dChIJ...\\
    """
    if not html_source:
        print("  No HTML source provided")
        return None
    
    # Pattern 1: placeid\\u003dChIJ...\\ (the pattern you mentioned)
    pattern1 = r'placeid\\u003d([A-Za-z0-9_-]+)\\\\'
    matches1 = re.findall(pattern1, html_source)
    
    if matches1:
        print(f"  Found {len(matches1)} place IDs with pattern 1")
        for match in matches1[:3]:  # Show first 3 matches
            if match.startswith('ChIJ'):
                print(f"    ✓ Found place ID: {match}")
                return match
        # Return first match even if not ChIJ
        return matches1[0]
    
    # Pattern 2: ChIJ followed by alphanumeric (27 chars)
    pattern2 = r'ChIJ[A-Za-z0-9_-]{27}'
    matches2 = re.findall(pattern2, html_source)
    
    if matches2:
        print(f"  Found {len(matches2)} place IDs with pattern 2")
        for match in matches2[:3]:
            print(f"    ✓ Found place ID: {match}")
        return matches2[0]
    
    # Pattern 3: Any 27-character alphanumeric string
    pattern3 = r'[A-Za-z0-9_-]{27}'
    matches3 = re.findall(pattern3, html_source)
    
    if matches3:
        print(f"  Found {len(matches3)} potential place IDs with pattern 3")
        for match in matches3[:5]:
            if match.startswith('ChIJ'):
                print(f"    ✓ Found place ID: {match}")
                return match
    
    print(f"  ✗ No place IDs found in HTML")
    return None

def find_place_page_url_in_search(html_source, current_url):
    """
    Find Google Maps place page URL in search results
    """
    if not html_source:
        return None
    
    # Check if we're already on a place page
    if is_place_page_url(current_url):
        return current_url
    
    # Look for place page URLs in search results
    patterns = [
        r'https://www\.google\.com/maps/place/[^\s"\']*?@[-+]?\d+\.\d+,[-+]?\d+\.\d+[^\s"\']*',
        r'/maps/place/[^\s"\']*?@[-+]?\d+\.\d+,[-+]?\d+\.\d+[^\s"\']*',
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, html_source)
        if matches:
            # Get the first match
            url = matches[0]
            if url.startswith('/maps/'):
                return f"https://www.google.com{url}"
            else:
                return url
    
    return None

def get_place_page_info(row, driver, index, save_html=True):
    """
    Get Google Maps place page URL, coordinates, and place ID
    """
    try:
        pool_name = row.get('pool_name', '').strip()
        address = row.get('address_admin', '').strip()
        pool_id = row.get('pool_id', '')
        
        if not pool_name or not address:
            print(f"  ✗ Missing pool_name or address_admin for pool_id {pool_id}")
            return '', '', '', '', 'missing_data', ''
        
        # Generate search URL
        search_url = generate_search_url(pool_name, address)
        
        print(f"\n{'='*60}")
        print(f"Row {index} (Pool ID: {pool_id}):")
        print(f"  Pool: {pool_name}")
        print(f"  Address: {address}")
        print(f"  Generated search URL: {search_url}")
        
        # Navigate to search URL
        print(f"  Navigating to search URL...")
        driver.get(search_url)
        time.sleep(5)  # Wait for page to load
        
        # Get current URL and page source
        current_url = driver.current_url
        search_page_source = driver.page_source
        
        print(f"  Current URL after search: {current_url}")
        
        # Try to find place page URL
        place_page_url = find_place_page_url_in_search(search_page_source, current_url)
        
        if not place_page_url:
            print(f"  ✗ Could not find place page URL in search results")
            return '', '', '', '', 'no_place_page_found', ''
        
        print(f"  Found place page URL: {place_page_url}")
        
        # Navigate to place page if not already there
        if place_page_url != current_url:
            print(f"  Navigating to place page...")
            driver.get(place_page_url)
            time.sleep(5)  # Wait for place page to load
        
        # Get place page details
        place_page_url = driver.current_url  # In case of redirect
        place_page_source = driver.page_source
        
        print(f"  Final place page URL: {place_page_url}")
        print(f"  Place page source length: {len(place_page_source):,} characters")
        
        # Extract coordinates from URL
        lat, lng = extract_coordinates_from_url(place_page_url)
        if lat and lng:
            print(f"  ✓ Coordinates: {lat}, {lng}")
        else:
            print(f"  ✗ Could not extract coordinates from URL")
        
        # Extract place ID from HTML
        place_id = extract_place_id_from_html(place_page_source)
        
        # Save HTML of place page
        html_file = ''
        if save_html:
            html_file = save_place_page_html(place_page_source, place_page_url, index)
        
        return place_page_url, lat, lng, place_id, 'success', html_file
    
    except WebDriverException as e:
        print(f"  ✗ WebDriver error: {str(e)}")
        return '', '', '', '', f'driver_error: {str(e)}', ''
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return '', '', '', '', f'error: {str(e)}', ''

def process_csv(input_file, output_file, save_html=True):
    """
    Process CSV file with pool data and extract place page information
    """
    # Get absolute paths using current working directory instead of __file__
    current_dir = os.getcwd()
    input_path = os.path.abspath(input_file)
    output_path = os.path.abspath(output_file)
    
    print(f"Current directory: {current_dir}")
    print(f"Input file: {input_path}")
    print(f"Output file: {output_path}")
    
    driver = setup_driver()
    
    try:
        with open(input_file, 'r', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            original_fieldnames = reader.fieldnames
            
            print(f"Input CSV columns: {original_fieldnames}")
            
            # Create output fieldnames
            output_fieldnames = list(original_fieldnames) + [
                'place_page_url',
                'latitude',
                'longitude',
                'place_id',
                'status',
                'html_file'
            ]
            
            with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
                writer = csv.DictWriter(outfile, fieldnames=output_fieldnames)
                writer.writeheader()
                
                row_count = 0
                success_count = 0
                place_id_count = 0
                
                for i, row in enumerate(reader):
                    row_count += 1
                    
                    print(f"\nProcessing row {i+1} (Pool ID: {row.get('pool_id', 'N/A')})...")
                    
                    # Get place page info
                    place_url, lat, lng, place_id, status, html_file = get_place_page_info(
                        row, driver, i+1, save_html
                    )
                    
                    # Add new data to row
                    row['place_page_url'] = place_url
                    row['latitude'] = lat if lat is not None else ''
                    row['longitude'] = lng if lng is not None else ''
                    row['place_id'] = place_id if place_id else ''
                    row['status'] = status
                    row['html_file'] = html_file
                    
                    writer.writerow(row)
                    
                    # Track success
                    if 'success' in status:
                        success_count += 1
                    if place_id:
                        place_id_count += 1
                    
                    print(f"  Status: {status}")
                    print(f"  Place ID found: {'Yes' if place_id else 'No'}")
                    print(f"{'='*60}")
                    
                    # Wait between requests
                    time.sleep(3)
                
                print(f"\nProcessing complete!")
                print(f"Total rows processed: {row_count}")
                print(f"Successful place page extractions: {success_count}")
                print(f"Place IDs found: {place_id_count}")
                print(f"Success rate: {success_count/row_count*100:.1f}%")
                print(f"Place ID rate: {place_id_count/row_count*100:.1f}%")
    
    except Exception as e:
        print(f"Fatal error: {str(e)}")
        import traceback
        traceback.print_exc()
    
    finally:
        driver.quit()
        print("\nBrowser closed")
        print(f"\nOutput saved to: {output_path}")
        
        if save_html:
            saved_pages_dir = os.path.abspath("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/saved_place_pages")
            print(f"Place page HTML files saved to: {saved_pages_dir}")
            if os.path.exists(saved_pages_dir):
                html_files = [f for f in os.listdir(saved_pages_dir) if f.endswith('.html')]
                print(f"Number of place page HTML files saved: {len(html_files)}")

def search_html_for_place_ids(html_directory="saved_place_pages"):
    """
    Search all saved HTML files for place IDs
    """
    html_directory = os.path.abspath(html_directory)
    
    if not os.path.exists(html_directory):
        print(f"Directory '{html_directory}' does not exist!")
        return {}
    
    html_files = [f for f in os.listdir(html_directory) if f.endswith('.html')]
    
    print(f"\nSearching {len(html_files)} saved HTML files for place IDs...")
    print(f"Directory: {html_directory}")
    
    results = {}
    
    for filename in html_files:
        filepath = os.path.join(html_directory, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            # Look for the specific pattern: placeid\\u003dChIJ...\\
            pattern = r'placeid\\u003d([A-Za-z0-9_-]+)\\\\'
            matches = re.findall(pattern, html_content)
            
            if matches:
                # Filter for ChIJ place IDs
                chij_matches = [m for m in matches if m.startswith('ChIJ')]
                if chij_matches:
                    place_id = chij_matches[0]
                    results[filename] = place_id
                    print(f"  ✓ {filename}: {place_id}")
                else:
                    # If no ChIJ, take the first match
                    results[filename] = matches[0]
                    print(f"  ✓ {filename}: {matches[0]} (not ChIJ)")
            else:
                print(f"  ✗ {filename}: No place ID found with pattern")
                
        except Exception as e:
            print(f"  Error reading {filename}: {str(e)}")
    
    # Save results to CSV
    if results:
        results_file = os.path.join(html_directory, "place_ids_found.csv")
        with open(results_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['filename', 'place_id'])
            for filename, place_id in results.items():
                writer.writerow([filename, place_id])
        print(f"\nPlace IDs summary saved to: {results_file}")
    
    return results

# Helper function to examine a specific HTML file
def examine_html_file(filepath):
    """
    Examine an HTML file to see what's in it
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        
        print(f"\nExamining: {filepath}")
        print(f"File size: {len(content)} characters")
        
        # Look for the placeid pattern
        pattern = r'placeid\\u003d([A-Za-z0-9_-]+)\\\\'
        matches = re.findall(pattern, content)
        
        if matches:
            print(f"Found {len(matches)} place IDs with pattern:")
            for i, match in enumerate(matches):
                print(f"  {i+1}. {match}")
        else:
            print(f"No place IDs found with pattern")
            
        # Also look for ChIJ pattern
        chij_pattern = r'ChIJ[A-Za-z0-9_-]{27}'
        chij_matches = re.findall(chij_pattern, content)
        
        if chij_matches:
            print(f"Found {len(chij_matches)} ChIJ patterns:")
            for i, match in enumerate(chij_matches[:10]):
                print(f"  {i+1}. {match}")
                
    except Exception as e:
        print(f"Error examining file: {str(e)}")

# Main execution
if __name__ == "__main__":
    # Configuration
    INPUT_CSV = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/843_pools.csv"
    OUTPUT_CSV = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/843_pools_pids.csv"
    
    # Set to True to save HTML files of place pages
    SAVE_HTML_FILES = True
    
    # Run the processing
    print("Starting Google Maps place page extraction...")
    print(f"This will save HTML files of Google Maps place pages")
    print(f"and extract place IDs using pattern: placeid\\\\u003dChIJ...\\\\")
    
    process_csv(
        input_file=INPUT_CSV,
        output_file=OUTPUT_CSV,
        save_html=SAVE_HTML_FILES
    )
    
    # After processing, search the saved HTML files for place IDs
    if SAVE_HTML_FILES and os.path.exists("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/saved_place_pages"):
        search_html_for_place_ids()

In [ ]:
###successfull files
success_url = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/843_pools_pids.csv")
success_url = success_url[success_url['status']=='success'].drop(columns=['place_id', 'html_file'])
success_url.to_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/843_pools_success_urls.csv", index=False)


In [ ]:
success_url